In [ ]:
# Task 1.1 — Read the file the foundations way
import csv
import json
import numpy as np
import pandas as pd

data = pd.read_csv("/content/appointments.csv")

def parse_row(row):
  """Converts specific string fields in a row dictionary to int or float."""
  parsed = row.copy()
  parsed["age"] = int(row["age"])
  parsed["fee_inr"] = int(row["fee_inr"])
  parsed["prior_no_shows"] = int(row["prior_no_shows"])
  parsed["distance_km"] = float(row["distance_km"])
  return parsed

accepted = []
rejected = []

with open("appointments.csv", mode="r", newline="", encoding="utf-8") as f:
  reader = csv.DictReader(f)
  headers = reader.fieldnames

  for row in reader:
    try:
      clean_row = parse_row(row)
      accepted.append(clean_row)
    except ValueError:
      rejected.append(row)

total_rows = len(accepted) + len(rejected)
print(f"Header: {headers}")
print(f"Row count: {total_rows}")

with open("rejected_rows.json", mode="w", encoding="utf-8") as out_file:
  json.dump(rejected, out_file, indent=2)



In [ ]:
# Task 1.2 — Inspect and clean, pass

# Loading the original dataset
df = pd.read_csv("/content/appointments.csv")

# Initial inspection
print("Shape:", df.shape)
print("\nInfo:")
df.info()
print("\nMissing values per column:\n", df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("-" * 40)

# (a) Drop exact duplicate rows
df = df.drop_duplicates()
print(f"Row count after (a) dropping duplicates: {len(df)}")

# (b) Clean booking_channel
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print(f"Row count after (b) cleaning booking_channel: {len(df)}")

# (c) Convert date columns
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print(f"Row count after (c) converting dates: {len(df)}")

# (d) Drop rows where appt_date is missing after conversion or status is missing
df = df.dropna(subset=["appt_date", "status"])
print(f"Row count after (d) dropping missing appt_date or status: {len(df)}")




In [ ]:
# Task 1.3 — Clean, pass 2: validity, gaps and outliers

import numpy as np
import pandas as pd

# Assuming your DataFrame is named `df`

# (a) Treat any age below 1 or above 100 as invalid and set it to missing
df.loc[(df['age'] < 1) | (df['age'] > 100), 'age'] = np.nan

# (b) Fill missing age with the median (as int) and missing distance_km with its median
df['age'] = df['age'].fillna(df['age'].median()).round().astype(int)
df['distance_km'] = df['distance_km'].fillna(df['distance_km'].median())

# (c) Compute IQR fences and cap outliers using np.clip for distance_km and fee_inr
for col in ['distance_km', 'fee_inr']:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    # Count values falling outside the fences before clipping
    capped_count = ((df[col] < lower_fence) | (df[col] > upper_fence)).sum()

    df[col] = np.clip(df[col], lower_fence, upper_fence)
    print(f"Values capped in {col}: {capped_count}")

# (d) Round distance_km to 1 decimal and fee_inr to whole rupees
df['distance_km'] = df['distance_km'].round(1)
df['fee_inr'] = df['fee_inr'].round().astype(int)


In [ ]:
# Task 1.4 — Features and a NumPy summary

import numpy as np
import pandas as pd

# Ensure date columns are in datetime format before subtracting
df['booked_on'] = pd.to_datetime(df['booked_on'])
df['appt_date'] = pd.to_datetime(df['appt_date'])

# Create lead_days and no_show columns
df['lead_days'] = (df['appt_date'] - df['booked_on']).dt.days
df['no_show'] = np.where(df['status'] == 'No-Show', 1, 0)

# Convert selected features to a 2D NumPy array
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) Mean of each column (axis=0)
col_means = X.mean(axis=0)
print("Column means (age, distance_km, fee_inr, lead_days):", col_means)

# (b) Number and share of rows with lead_days > 14 (lead_days is at index 3)
lead_mask = X[:, 3] > 14
count_gt_14 = lead_mask.sum()
share_gt_14 = lead_mask.mean()
print(f"Rows with lead_days > 14: {count_gt_14} (Share: {share_gt_14:.4f})")

# (c) Z-scores of distance_km (index 1) and count of rows with |z| > 2
dist_col = X[:, 1]
z_scores = (dist_col - np.mean(dist_col)) / np.std(dist_col)
outliers_count = (np.abs(z_scores) > 2).sum()
print(f"Rows with |z| > 2 for distance_km: {outliers_count}")


In [ ]:
# Task 1.5 — Enrich the clinics from a live API
import time
import pandas as pd
import requests


def fetch_clinic_temp(name, lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m",
        "forecast_days": 1,
    }

    last_reason = "Unknown error"

    # Initial attempt + 1 retry (2 attempts total)
    for attempt in range(2):
        try:
            response = requests.get(url, params=params, timeout=10)

            if response.status_code != 200:
                last_reason = f"HTTP status {response.status_code}"
            else:
                data = response.json()
                hourly = data.get("hourly")

                if (
                    not isinstance(hourly, dict)
                    or "temperature_2m" not in hourly
                ):
                    last_reason = "Missing 'hourly' or 'temperature_2m' in JSON"
                elif (
                    not hourly["temperature_2m"]
                    or len(hourly["temperature_2m"]) == 0
                ):
                    last_reason = "'temperature_2m' list is empty"
                else:
                    return hourly["temperature_2m"][0]

        except requests.exceptions.RequestException as e:
            last_reason = f"RequestException ({e})"

        # Brief pause before retrying once
        if attempt == 0:
            time.sleep(1)

    print(f"Failed for {name}: {last_reason}")
    return None


# Load original clinic data
df = pd.read_csv("/content/clinics.csv")

# Fetch temperature for each clinic (using 'clinic_name' instead of 'name')
df["temp_c"] = df.apply(
    lambda row: fetch_clinic_temp(row["clinic_name"], row["latitude"], row["longitude"]),
    axis=1,
)

# Save enriched CSV (None/NaN writes as an empty field in CSV)
df.to_csv("clinics_enriched.csv", index=False)
print("Saved clinics_enriched.csv")
